In [2]:
# ============================================================
# IMPORTAÇÕES
# ============================================================

import time
import itertools
import numpy as np
import pandas as pd

from tqdm.auto import tqdm

from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
from sklearn.metrics import (
    precision_recall_curve,
    auc,
    log_loss
)


# ============================================================
# FUNÇÃO PARA TRUNCAR EM 6 CASAS DECIMAIS
# ============================================================

def truncar_6(x):
    """
    Trunca um número em 6 casas decimais, sem arredondar.
    """
    return np.trunc(float(x) * 1_000_000) / 1_000_000


# ============================================================
# LEITURA DO CSV
# ============================================================

df = pd.read_csv("creditcard.csv")


# ============================================================
# DEFINIÇÃO DO TARGET
# ============================================================

if "status_fraude" in df.columns:
    target_name = "status_fraude"

elif "Class" in df.columns:
    df = df.rename(columns={"Class": "status_fraude"})
    target_name = "status_fraude"

else:
    raise ValueError("Não encontrei a coluna target: 'status_fraude' ou 'Class'.")


# ============================================================
# SELEÇÃO DAS FEATURES NUMÉRICAS
# ============================================================

features = [
    col for col in df.columns
    if col != target_name
    and pd.api.types.is_numeric_dtype(df[col])
]


# ============================================================
# COMBINAÇÕES 3x3
# ============================================================

combinacoes_3x3 = list(itertools.combinations(features, 3))

print("Dataset carregado com sucesso.")
print(f"Shape do dataset: {df.shape}")
print(f"Target utilizado: {target_name}")
print(f"Quantidade de features numéricas: {len(features)}")
print(f"Quantidade de combinações 3x3: {len(combinacoes_3x3)}")


# ============================================================
# LOG-PDF GAUSSIANA MULTIVARIADA
# ============================================================

def logpdf_gaussiana_multivariada(X, media, cov):
    """
    Calcula log N(x | media, cov).

    Funciona para 1D e múltiplas dimensões.
    No caso 3x3, X terá três colunas.
    """

    X = np.asarray(X)
    media = np.asarray(media)
    cov = np.asarray(cov)

    if X.ndim == 1:
        X = X.reshape(-1, 1)

    if media.ndim == 0:
        media = np.array([media])

    media = media.reshape(-1)

    cov = np.atleast_2d(cov)

    n_features = X.shape[1]

    sinal, logdet = np.linalg.slogdet(cov)

    if sinal <= 0:
        return np.full(X.shape[0], -np.inf)

    diff = X - media

    solucao = np.linalg.solve(cov, diff.T).T

    termo_quadratico = np.sum(
        diff * solucao,
        axis=1
    )

    logpdf = -0.5 * (
        n_features * np.log(2 * np.pi)
        + logdet
        + termo_quadratico
    )

    return logpdf


# ============================================================
# LOG-VEROSSIMILHANÇA COM RÓTULO
# ============================================================

def calcular_log_veross_com_rotulo(
    X_scaled,
    y_real,
    reg_covar=1e-6
):
    """
    Calcula a log-verossimilhança média por amostra usando o rótulo real.

    Ideia:
    - y = 0 define uma Gaussiana para não fraude
    - y = 1 define uma Gaussiana para fraude
    - os pesos são as proporções reais das classes

    Retorna:
    - log-verossimilhança média por amostra.
    """

    X_scaled = np.asarray(X_scaled)

    if X_scaled.ndim == 1:
        X_scaled = X_scaled.reshape(-1, 1)

    y_real = np.asarray(y_real).astype(int)

    n_amostras, n_features = X_scaled.shape

    log_veross_total = 0.0

    for classe in [0, 1]:

        X_classe = X_scaled[y_real == classe]

        n_classe = X_classe.shape[0]

        if n_classe <= 1:
            return np.nan

        peso_classe = n_classe / n_amostras

        media_classe = np.mean(
            X_classe,
            axis=0
        )

        cov_classe = np.cov(
            X_classe,
            rowvar=False
        )

        cov_classe = np.atleast_2d(cov_classe)

        cov_classe = cov_classe + reg_covar * np.eye(n_features)

        logpdf_classe = logpdf_gaussiana_multivariada(
            X=X_classe,
            media=media_classe,
            cov=cov_classe
        )

        log_veross_total += np.sum(
            np.log(peso_classe) + logpdf_classe
        )

    log_veross_media = log_veross_total / n_amostras

    return log_veross_media


# ============================================================
# FUNÇÃO OTIMIZADA PARA ENCONTRAR O MELHOR PONTO DE CORTE
# PELO MCC USANDO AS PRÓPRIAS PROBABILIDADES COMO THRESHOLDS
# ============================================================

def encontrar_melhor_ponto_corte_mcc(y_real, probabilidades):
    """
    Encontra o melhor ponto de corte pelo MCC.

    Usa como thresholds as próprias probabilidades estimadas pelo modelo,
    mas calcula tudo de forma otimizada via ordenação e somas acumuladas.

    Regra:
        y_pred = 1 se probabilidade >= threshold
        y_pred = 0 caso contrário
    """

    y_real = np.asarray(y_real).astype(int)
    probabilidades = np.asarray(probabilidades)

    ordem = np.argsort(-probabilidades)

    probs_ord = probabilidades[ordem]
    y_ord = y_real[ordem]

    total_positivos = np.sum(y_ord == 1)
    total_negativos = np.sum(y_ord == 0)

    tp_acum = np.cumsum(y_ord == 1)
    fp_acum = np.cumsum(y_ord == 0)

    fn_acum = total_positivos - tp_acum
    tn_acum = total_negativos - fp_acum

    numerador = (tp_acum * tn_acum) - (fp_acum * fn_acum)

    denominador = np.sqrt(
        (tp_acum + fp_acum) *
        (tp_acum + fn_acum) *
        (tn_acum + fp_acum) *
        (tn_acum + fn_acum)
    )

    mccs = np.divide(
        numerador,
        denominador,
        out=np.zeros_like(numerador, dtype=float),
        where=denominador != 0
    )

    indices_validos = np.r_[
        np.where(probs_ord[:-1] != probs_ord[1:])[0],
        len(probs_ord) - 1
    ]

    mccs_validos = mccs[indices_validos]

    melhor_idx_local = np.argmax(mccs_validos)
    melhor_idx = indices_validos[melhor_idx_local]

    melhor_ponto_corte = probs_ord[melhor_idx]
    melhor_mcc = mccs[melhor_idx]

    return melhor_ponto_corte, melhor_mcc


# ============================================================
# FUNÇÃO PARA ANALISAR UMA COMBINAÇÃO 3x3
# ============================================================

def analisar_combinacao_3x3(
    df,
    feature_1,
    feature_2,
    feature_3,
    target_name="status_fraude",
    resumo_combinacoes=None,
    verbose=False
):

    if resumo_combinacoes is None:
        resumo_combinacoes = []

    inicio = time.perf_counter()

    if verbose:
        print(f"\nIniciando combinação: {feature_1} + {feature_2} + {feature_3}")

    # ========================================================
    # DADOS
    # ========================================================

    temp = df[[feature_1, feature_2, feature_3, target_name]].dropna()

    if temp.empty:
        if verbose:
            print("  - Ignorada: dados vazios após dropna")
        return resumo_combinacoes

    X = temp[[feature_1, feature_2, feature_3]]
    y_real = temp[target_name].astype(int)

    if y_real.nunique() < 2:
        if verbose:
            print("  - Ignorada: target possui apenas uma classe")
        return resumo_combinacoes

    if (
        X[feature_1].nunique() < 2
        or X[feature_2].nunique() < 2
        or X[feature_3].nunique() < 2
    ):
        if verbose:
            print("  - Ignorada: uma das features é constante")
        return resumo_combinacoes

    # ========================================================
    # ESCALONAMENTO
    # ========================================================

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # ========================================================
    # TREINAMENTO GMM 3D
    # ========================================================

    try:
        gmm = GaussianMixture(
            n_components=2,
            covariance_type="full",
            random_state=42,
            n_init=3,
            reg_covar=1e-6
        )

        gmm.fit(X_scaled)

    except Exception as erro:
        if verbose:
            print(f"  - Erro no treinamento da GMM: {erro}")
        return resumo_combinacoes

    # ========================================================
    # LOG-VEROSSIMILHANÇA DA GMM SEM RÓTULO
    # ========================================================
    # gmm.score(X_scaled) retorna a log-verossimilhança média
    # por amostra segundo a GMM ajustada sem usar o rótulo.

    log_veross_gmm = gmm.score(X_scaled)

    # ========================================================
    # LOG-VEROSSIMILHANÇA COM RÓTULO
    # ========================================================
    # Aqui o rótulo real define as duas Gaussianas:
    # uma para y=0 e outra para y=1.

    log_veross_com_rotulo = calcular_log_veross_com_rotulo(
        X_scaled=X_scaled,
        y_real=y_real,
        reg_covar=1e-6
    )

    # ========================================================
    # NEGATIVE LOG-LIKELIHOODS
    # ========================================================
    # Agora trabalhamos com -log-verossimilhança.
    # Nesse caso, menor é melhor.

    neg_log_veross_com_rotulo = -log_veross_com_rotulo
    neg_log_veross_gmm = -log_veross_gmm

    diferenca_neg_log_veross = (
        neg_log_veross_com_rotulo
        - neg_log_veross_gmm
    )

    # ========================================================
    # IDENTIFICAR CLUSTER ASSOCIADO À FRAUDE
    # ========================================================

    clusters = gmm.predict(X_scaled)

    ct = pd.crosstab(
        clusters,
        y_real
    )

    if 1 not in ct.columns:
        if verbose:
            print("  - Ignorada: classe 1 não encontrada no crosstab")
        return resumo_combinacoes

    cluster_fraude = ct[1].idxmax()

    # ========================================================
    # PROBABILIDADES DO CLUSTER ASSOCIADO À FRAUDE
    # ========================================================

    probabilidades = gmm.predict_proba(X_scaled)[:, cluster_fraude]

    probabilidades = np.clip(
        probabilidades,
        1e-15,
        1 - 1e-15
    )

    # ========================================================
    # AUC-PR
    # ========================================================

    precision_vals, recall_vals, _ = precision_recall_curve(
        y_real,
        probabilidades
    )

    auc_pr = auc(
        recall_vals,
        precision_vals
    )

    # ========================================================
    # MELHOR PONTO DE CORTE E MCC
    # ========================================================

    melhor_ponto_corte, mcc = encontrar_melhor_ponto_corte_mcc(
        y_real=y_real,
        probabilidades=probabilidades
    )

    # ========================================================
    # PONTO DE CORTE MÉDIO
    # ========================================================

    ponto_corte_medio = 0.5

    # ========================================================
    # LOG LOSS
    # ========================================================

    ll = log_loss(
        y_real,
        probabilidades
    )

    fim = time.perf_counter()

    # ========================================================
    # NORMALIZAÇÕES
    # ========================================================

    auc_pr_norm = np.clip(
        auc_pr,
        0,
        1
    )

    mcc_norm = (mcc + 1) / 2

    mcc_norm = np.clip(
        mcc_norm,
        0,
        1
    )

    log_loss_norm = 1 / (1 + ll)

    log_loss_norm = np.clip(
        log_loss_norm,
        0,
        1
    )

    # ========================================================
    # SCORE FINAL
    # ========================================================

    score_final = np.mean([
        auc_pr_norm,
        mcc_norm,
        log_loss_norm
    ])

    # ========================================================
    # RESULTADO COM TRUNCAMENTO EM 6 CASAS
    # ========================================================

    nova_linha = {
        "Feature_1": feature_1,
        "Feature_2": feature_2,
        "Feature_3": feature_3,
        "Combinacao": f"{feature_1} + {feature_2} + {feature_3}",

        "AUC_PR": truncar_6(float(auc_pr)),
        "MCC": truncar_6(float(mcc)),
        "Log_Loss": truncar_6(float(ll)),
        "Log_Loss_Norm": truncar_6(float(log_loss_norm)),

        "Neg_Log_Veross_Com_Rotulo": truncar_6(float(neg_log_veross_com_rotulo)),
        "Neg_Log_Veross_GMM": truncar_6(float(neg_log_veross_gmm)),
        "Diferenca_Neg_Log_Veross": truncar_6(float(diferenca_neg_log_veross)),

        "Score_Final": truncar_6(float(score_final)),
        "Melhor_Ponto_Corte": truncar_6(float(melhor_ponto_corte)),
        "Ponto_Corte_Medio": truncar_6(float(ponto_corte_medio)),
        "Tempo": truncar_6(float(fim - inicio))
    }

    resumo_combinacoes.append(nova_linha)

    if verbose:
        print(f"  - Finalizada em {fim - inicio:.2f} segundos")
        print(f"  - Score_Final: {score_final:.6f}")
        print(f"  - Neg_Log_Veross_Com_Rotulo: {neg_log_veross_com_rotulo:.6f}")
        print(f"  - Neg_Log_Veross_GMM: {neg_log_veross_gmm:.6f}")
        print(f"  - Diferenca_Neg_Log_Veross: {diferenca_neg_log_veross:.6f}")

    return resumo_combinacoes


# ============================================================
# EXECUÇÃO PARA TODAS AS COMBINAÇÕES 3x3
# ============================================================

resumo_combinacoes = []

inicio_geral = time.perf_counter()

for feature_1, feature_2, feature_3 in tqdm(
    combinacoes_3x3,
    desc="Processando combinações 3x3",
    unit="combinação"
):
    tamanho_antes = len(resumo_combinacoes)

    resumo_combinacoes = analisar_combinacao_3x3(
        df=df,
        feature_1=feature_1,
        feature_2=feature_2,
        feature_3=feature_3,
        target_name=target_name,
        resumo_combinacoes=resumo_combinacoes,
        verbose=False
    )

    tamanho_depois = len(resumo_combinacoes)

    if tamanho_depois > tamanho_antes:
        tqdm.write(f"Combinação processada: {feature_1} + {feature_2} + {feature_3}")
    else:
        tqdm.write(f"Combinação ignorada ou com erro: {feature_1} + {feature_2} + {feature_3}")

fim_geral = time.perf_counter()


# ============================================================
# DATAFRAME FINAL
# ============================================================

scores_3x3 = pd.DataFrame(resumo_combinacoes)

if scores_3x3.empty:
    raise ValueError(
        "Nenhuma combinação 3x3 foi processada. "
        "Verifique se existem features numéricas válidas e se o target está correto."
    )

scores_3x3 = scores_3x3.sort_values(
    by="Score_Final",
    ascending=False
).reset_index(drop=True)

scores_3x3["Posicao_Rank"] = np.arange(
    1,
    len(scores_3x3) + 1
)

scores_3x3 = scores_3x3[
    [
        "Feature_1",
        "Feature_2",
        "Feature_3",
        "Combinacao",
        "AUC_PR",
        "MCC",
        "Log_Loss",
        "Log_Loss_Norm",
        "Neg_Log_Veross_Com_Rotulo",
        "Neg_Log_Veross_GMM",
        "Diferenca_Neg_Log_Veross",
        "Score_Final",
        "Melhor_Ponto_Corte",
        "Ponto_Corte_Medio",
        "Tempo",
        "Posicao_Rank"
    ]
]


# ============================================================
# EXPORTAÇÃO PARA CSV
# ============================================================

scores_3x3.to_csv(
    "3x3_scores.csv",
    index=False,
    encoding="utf-8-sig",
    float_format="%.6f"
)


# ============================================================
# RELATÓRIO FINAL
# ============================================================

tempo_total_segundos = fim_geral - inicio_geral
tempo_total_minutos = tempo_total_segundos / 60

print("\nProcessamento finalizado.")
print(f"Combinações processadas com sucesso: {len(scores_3x3)}")
print(f"Tempo total: {tempo_total_segundos:.2f} segundos")
print(f"Tempo total: {tempo_total_minutos:.2f} minutos")
print("Arquivo salvo como: 3x3_scores.csv")

print("\nTop 20 combinações:")
display(scores_3x3.head(20))

Dataset carregado com sucesso.
Shape do dataset: (283726, 31)
Target utilizado: status_fraude
Quantidade de features numéricas: 30
Quantidade de combinações 3x3: 4060


Processando combinações 3x3:   0%|          | 0/4060 [00:00<?, ?combinação/s]

Combinação processada: tempo_desde_a_primeira_transacao + V1 + V2
Combinação processada: tempo_desde_a_primeira_transacao + V1 + V3
Combinação processada: tempo_desde_a_primeira_transacao + V1 + V4
Combinação processada: tempo_desde_a_primeira_transacao + V1 + V5
Combinação processada: tempo_desde_a_primeira_transacao + V1 + V6
Combinação processada: tempo_desde_a_primeira_transacao + V1 + V7
Combinação processada: tempo_desde_a_primeira_transacao + V1 + V8
Combinação processada: tempo_desde_a_primeira_transacao + V1 + V9
Combinação processada: tempo_desde_a_primeira_transacao + V1 + V10
Combinação processada: tempo_desde_a_primeira_transacao + V1 + V11
Combinação processada: tempo_desde_a_primeira_transacao + V1 + V12
Combinação processada: tempo_desde_a_primeira_transacao + V1 + V13
Combinação processada: tempo_desde_a_primeira_transacao + V1 + V14
Combinação processada: tempo_desde_a_primeira_transacao + V1 + V15
Combinação processada: tempo_desde_a_primeira_transacao + V1 + V16
Com

,Feature_1,Feature_2,Feature_3,Combinacao,AUC_PR,MCC,Log_Loss,Log_Loss_Norm,Neg_Log_Veross_Com_Rotulo,Neg_Log_Veross_GMM,Diferenca_Neg_Log_Veross,Score_Final,Melhor_Ponto_Corte,Ponto_Corte_Medio,Tempo,Posicao_Rank
0,V11,V17,V26,V11 + V17 + V26,0.559764,0.593525,0.122908,0.890544,4.135136,4.060821,0.074314,0.749023,0.999999,0.5,13.481611,1
1,V11,V17,V22,V11 + V17 + V22,0.542839,0.575421,0.113745,0.897871,4.134605,4.052167,0.082438,0.742807,0.999999,0.5,13.770865,2
2,V11,V14,V16,V11 + V14 + V16,0.626360,0.644598,0.304941,0.766317,4.152508,4.044957,0.107551,0.738325,0.999999,0.5,11.608719,3
3,V14,V17,V25,V14 + V17 + V25,0.624276,0.636269,0.294824,0.772305,4.079120,3.890455,0.188665,0.738239,0.999999,0.5,7.321723,4
4,V11,V13,V17,V11 + V13 + V17,0.552266,0.585630,0.151309,0.868575,4.135153,4.060835,0.074317,0.737885,0.999999,0.5,13.367419,5
5,V11,V17,V19,V11 + V17 + V19,0.546300,0.585659,0.153727,0.866755,4.132600,4.037575,0.095024,0.735295,0.999999,0.5,9.493933,6
6,V3,V12,V14,V3 + V12 + V14,0.602831,0.614920,0.272937,0.785584,4.112434,3.938171,0.174263,0.731958,0.999999,0.5,9.212685,7
7,V14,V16,V26,V14 + V16 + V26,0.630577,0.644544,0.346847,0.742474,4.172745,4.101456,0.071289,0.731774,0.999999,0.5,13.125599,8
8,V4,V11,V17,V4 + V11 + V17,0.578906,0.598419,0.231367,0.812104,4.121702,4.029939,0.091763,0.730073,0.999999,0.5,12.270021,9
9,V11,V17,V25,V11 + V17 + V25,0.538668,0.573177,0.161263,0.861130,4.134797,4.042421,0.092376,0.728795,0.999999,0.5,12.478652,10
